# MPC Path-Planning Demo (Toy)

This notebook focuses on MPC as a path-planning tool: local receding-horizon optimization for path tracking with obstacle awareness.

In [ ]:
%matplotlib inline

import math

import matplotlib.pyplot as plt
import numpy as np

## 1. Reference path, obstacles, and robot model

We use a unicycle-like model with state $[x, y, yaw]$ and controls $[v, w]$.

In [ ]:
dt = 0.15

def step(x, u):
    # x = [x, y, yaw], u = [v, w]
    xn = np.zeros_like(x)
    xn[2] = x[2] + u[1] * dt
    xn[0] = x[0] + u[0] * math.cos(xn[2]) * dt
    xn[1] = x[1] + u[0] * math.sin(xn[2]) * dt
    return xn

# Reference path in XY (global plan segment)
x_ref = np.linspace(0.5, 12.0, 260)
y_ref = 1.0 + 0.35 * x_ref + 0.9 * np.sin(0.55 * x_ref)
ref = np.column_stack([x_ref, y_ref])

obstacles = np.array([
    [4.5, 3.1], [5.4, 3.5], [7.2, 4.2], [8.4, 4.7], [9.6, 4.8]
])

x = np.array([0.5, 0.9, 0.0])

## 2. MPC setup: horizon, constraints, and costs

In [ ]:
N = 10
n_rollouts = 700

v_min, v_max = 0.0, 1.3
w_min, w_max = -1.2, 1.2

w_track = 2.2
w_heading = 0.35
w_ctrl = 0.08
w_rate = 0.10
w_obs = 2.4
safe_dist = 0.85

rng = np.random.default_rng(5)

## 3. Helper functions for reference indexing and costs

In [ ]:
def nearest_ref_index(pos):
    d = np.linalg.norm(ref - pos, axis=1)
    return int(np.argmin(d))

def heading_of_segment(i):
    j = min(i + 1, len(ref) - 1)
    dx, dy = ref[j, 0] - ref[i, 0], ref[j, 1] - ref[i, 1]
    return math.atan2(dy, dx)

def angle_wrap(a):
    return (a + math.pi) % (2 * math.pi) - math.pi

def obstacle_penalty(p):
    dmin = np.min(np.linalg.norm(obstacles - p, axis=1))
    if dmin >= safe_dist:
        return 0.0
    return (safe_dist - dmin) ** 2

## 4. Sample-based receding-horizon MPC (path-planning view)

For teaching clarity, we use random-shooting over bounded controls and pick the minimum-cost feasible sequence.

In [ ]:
def mpc_plan(x_now, u_prev):
    i0 = nearest_ref_index(x_now[:2])

    best_cost = np.inf
    best_u0 = np.array([0.0, 0.0])
    best_pred = None

    V = rng.uniform(v_min, v_max, size=(n_rollouts, N))
    W = rng.uniform(w_min, w_max, size=(n_rollouts, N))

    for r in range(n_rollouts):
        xk = x_now.copy()
        pred = [xk.copy()]
        cost = 0.0
        up = u_prev.copy()

        for k in range(N):
            u = np.array([V[r, k], W[r, k]])
            xk = step(xk, u)
            pred.append(xk.copy())

            # Local reference point ahead
            ir = min(i0 + 2 * (k + 1), len(ref) - 1)
            rxy = ref[ir]
            th_ref = heading_of_segment(ir)

            e_track = np.linalg.norm(xk[:2] - rxy) ** 2
            e_head = angle_wrap(xk[2] - th_ref) ** 2
            e_ctrl = u[0] ** 2 + 0.3 * u[1] ** 2
            e_rate = (u[0] - up[0]) ** 2 + (u[1] - up[1]) ** 2
            e_obs = obstacle_penalty(xk[:2])

            cost += w_track * e_track + w_heading * e_head
            cost += w_ctrl * e_ctrl + w_rate * e_rate + w_obs * e_obs

            up = u

        if cost < best_cost:
            best_cost = cost
            best_u0 = np.array([V[r, 0], W[r, 0]])
            best_pred = np.array(pred)

    return best_u0, best_pred, best_cost

## 5. Closed-loop planning and execution

In [ ]:
traj = [x.copy()]
u_prev = np.array([0.0, 0.0])
u_hist = []
preds = []

for _ in range(85):
    u_cmd, pred, J = mpc_plan(x, u_prev)
    x = step(x, u_cmd)

    traj.append(x.copy())
    u_hist.append(u_cmd.copy())
    if pred is not None:
        preds.append(pred)

    u_prev = u_cmd

    if np.linalg.norm(x[:2] - ref[-1]) < 0.5:
        break

traj = np.array(traj)
u_hist = np.array(u_hist)

print('Executed steps:', len(traj) - 1)
print('Distance to final ref point:', round(float(np.linalg.norm(traj[-1, :2] - ref[-1])), 3))

## 6. Visualize path-planning behavior

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ref[:, 0], ref[:, 1], '--', color='gray', linewidth=1.5, label='reference path')
ax.scatter(obstacles[:, 0], obstacles[:, 1], c='black', s=35, label='obstacles')

if preds:
    stride = max(1, len(preds) // 12)
    for p in preds[::stride]:
        ax.plot(p[:, 0], p[:, 1], color='slateblue', alpha=0.22, linewidth=1.0)

ax.plot(traj[:, 0], traj[:, 1], color='deepskyblue', linewidth=2.6, label='executed MPC trajectory')
ax.scatter(traj[0, 0], traj[0, 1], c='limegreen', s=80, marker='o', label='start')
ax.scatter(ref[-1, 0], ref[-1, 1], c='crimson', s=85, marker='*', label='goal target')

ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('MPC used as local path-planning/tracking optimizer')
ax.legend(loc='best')
plt.tight_layout()
plt.show()

## 7. Self-study experiments

1. Increase `w_obs` and check how aggressively the trajectory avoids obstacles.
2. Increase `w_track` and see whether obstacle clearance decreases.
3. Increase horizon `N` and compare smoothness vs runtime.
4. Move obstacles onto the path and observe local replanning behavior.